<a href="https://colab.research.google.com/github/BenYILDO/breast_cancer_ML/blob/main/MachineLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import plotly.express as px

from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, precision_score, accuracy_score, recall_score,
    balanced_accuracy_score, roc_auc_score, confusion_matrix,
    roc_curve
)



------------



# 1. Veri Setinin Yüklenmesi ve İncelenmesi
## 1.1 Veri Seti Yükleme


Öncelikle yüklenen veri setinin hangi bileşenlerden oluştuğunu görmek için `keys( )` fonksiyonu kullanılabilir.




In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()

data.keys()

dict_keys(['data', 'target', 'frame', 'target_names', 'DESCR', 'feature_names', 'filename', 'data_module'])

## 1.2 Veri Çerçevesi Oluşturma

In [ ]:
print("Özellik isimleri:")
print(data.feature_names)

print("\nSınıf isimleri:")
print(data.target_names)

print("\nVeri boyutu:")
print(data.data.shape)

print("\nHedef boyutu:")
print(data.target.shape)

Özellik isimleri:
['mean radius' 'mean texture' 'mean perimeter' 'mean area'
 'mean smoothness' 'mean compactness' 'mean concavity'
 'mean concave points' 'mean symmetry' 'mean fractal dimension'
 'radius error' 'texture error' 'perimeter error' 'area error'
 'smoothness error' 'compactness error' 'concavity error'
 'concave points error' 'symmetry error' 'fractal dimension error'
 'worst radius' 'worst texture' 'worst perimeter' 'worst area'
 'worst smoothness' 'worst compactness' 'worst concavity'
 'worst concave points' 'worst symmetry' 'worst fractal dimension']

Sınıf isimleri:
['malignant' 'benign']

Veri boyutu:
(569, 30)

Hedef boyutu:
(569,)


* data.data: bağımsız değişkenleri içerir
* data.target: hedef sınıfı içerir
* data.feature_names: sütun isimlerini verir
* data.target_names: sınıf isimlerini verir
* data.DESCR: veri setinin açıklamasını içerir



Bu çıktı, veri setinin yalnızca gözlem verilerinden oluşmadığını aynı zamanda hedef değişkeni, özellik isimlerini, sınıf isimlerini ve açıklama metnini de içerdiğini göstermektedir.

In [ ]:
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

In [44]:
df = pd.concat([X, y], axis=1)

df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


- Modelleme aşamasında bağımsız değişkenler `X`, hedef değişken ise `y` olarak ayrılmıştır.
- Keşifsel veri analizi yapabilmek için `X` ve `y` birleştirilerek `df` isimli yeni bir DataFrame oluşturulmuştur.
- Bu sayede özellikler ve hedef sınıf aynı tablo üzerinde birlikte incelenebilir.

In [45]:
print("X boyutu:", X.shape)
print("y boyutu:", y.shape)
print("df boyutu:", df.shape)

X boyutu: (569, 30)
y boyutu: (569,)
df boyutu: (569, 31)


- Veri setinde 569 gözlem ve 30 bağımsız değişken bulunmaktadır.
- Hedef değişken de eklendiğinde toplam sütun sayısı 31 olmaktadır.
- Bu problem, tümörün iyi huylu veya kötü huylu olmasını tahmin etmeye yönelik iki sınıflı bir sınıflandırma problemidir.








---



# 2. Veri Seti Kalite Kontrolleri

Bu bölümde veri setinde eksik değer olup olmadığı, değişkenlerin veri tipleri ve genel veri yapısı incelenmiştir.
Veri kalitesi kontrolü, modelleme aşamasına geçmeden önce veride hata, eksiklik veya uygunsuz format olup olmadığını anlamak için önemlidir.

In [46]:
df.isnull().sum()

,0
mean radius,0
mean texture,0
mean perimeter,0
mean area,0
mean smoothness,0
mean compactness,0
mean concavity,0
mean concave points,0
mean symmetry,0
mean fractal dimension,0


Yapılan eksik değer kontrolünde veri setindeki sütunlarda eksik değer bulunmadığı görülmüştür.
Bu nedenle herhangi bir eksik değer doldurma veya silme işlemi uygulanmamıştır.

In [51]:
df.dtypes

,0
mean radius,float64
mean texture,float64
mean perimeter,float64
mean area,float64
mean smoothness,float64
mean compactness,float64
mean concavity,float64
mean concave points,float64
mean symmetry,float64
mean fractal dimension,float64


In [55]:
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = df.select_dtypes(include=["object", "category"]).columns

print("Sayısal değişken sayısı:", len(numeric_cols))
print("Kategorik değişken sayısı:", len(categorical_cols))

Sayısal değişken sayısı: 31
Kategorik değişken sayısı: 0


Veri setindeki değişkenlerin tamamı sayısal yapıdadır.
Bu durum makine öğrenmesi modelleri açısından avantajlıdır çünkü ek bir kategorik değişken dönüştürme işlemine ihtiyaç duyulmamaktadır.
`target` değişkeni sayısal görünse de aslında sınıf bilgisini temsil etmektedir.